## Evaluate a custom Presidio Analyzer using the Presidio Evaluator framework

This notebook demonstrates how to evaluate a Presidio instance using the presidio-evaluator framework. It builds upon [notebook 4](4_Evaluate_Presidio_Analyzer.ipynb), customising the `PresidioAnalyzer` instance for better accuracy. For more on customising Presidio, see the [Presidio Analyzer documentation](https://microsoft.github.io/presidio/analyzer/).

Steps:
1. Load dataset
2. Dataset statistics
3. Define the custom Presidio Analyzer
4. Run predictions
5. Review and adjust entity mapping
6. Evaluate
7. Results and error analysis

In [10]:
# install presidio evaluator via pip if not yet installed

#!pip install presidio-evaluator
#!pip install "presidio-analyzer[transformers]"

In [11]:
import json
import warnings
from collections import Counter
from pathlib import Path
from pprint import pprint

warnings.filterwarnings('ignore')

import pandas as pd
from presidio_analyzer import (
    AnalyzerEngine,
    Pattern,
    PatternRecognizer,
    RecognizerRegistry,
)
from presidio_analyzer.context_aware_enhancers import LemmaContextAwareEnhancer
from presidio_analyzer.nlp_engine import NerModelConfiguration, TransformersNlpEngine

from presidio_evaluator import InputSample
from presidio_evaluator.entity_mapping import CanonicalMapper, IncompleteMapping
from presidio_evaluator.evaluation import ModelError, Plotter, SpanEvaluator
from presidio_evaluator.experiment_tracking import get_experiment_tracker
from presidio_evaluator.models import PresidioAnalyzerWrapper

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

%reload_ext autoreload
%autoreload 2
%matplotlib inline


## 1. Load dataset from file

In [12]:
dataset_name = "synth_dataset_v2.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd().parent, "data", dataset_name))

print(len(dataset))

tokenizing input: 100%|██████████| 1500/1500 [00:04<00:00, 332.29it/s]

1500


This dataset was auto generated. See more info here [Synthetic data generation](1_Generate_data.ipynb).

In [13]:
def get_entity_counts(dataset: list[InputSample]) -> dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter

## 2. Simple dataset statistics

In [14]:
entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print(
    "\nMin and max number of tokens in dataset: "
    f"Min: {min([len(sample.tokens) for sample in dataset])}, "
    f"Max: {max([len(sample.tokens) for sample in dataset])}"
)

print(
    f"Min and max sentence length in dataset: "
    f"Min: {min([len(sample.full_text) for sample in dataset])}, "
    f"Max: {max([len(sample.full_text) for sample in dataset])}"
)

Count per entity:
[('O', 19626), ('STREET_ADDRESS', 3071), ('PERSON', 1369), ('GPE', 521),
 ('ORGANIZATION', 504), ('PHONE_NUMBER', 350), ('DATE_TIME', 219),
 ('TITLE', 142), ('CREDIT_CARD', 136), ('US_SSN', 80), ('AGE', 74), ('NRP', 55),
 ('ZIP_CODE', 50), ('EMAIL_ADDRESS', 49), ('DOMAIN_NAME', 37),
 ('IP_ADDRESS', 22), ('IBAN_CODE', 21), ('US_DRIVER_LICENSE', 9)]

Min and max number of tokens in dataset: Min: 3, Max: 78
Min and max sentence length in dataset: Min: 9, Max: 407


## 3. Define the AnalyzerEngine object 
In this case, we customize the AnalyzerEngine to use a different NER model, some custom recognizers and the context aware enhancer.

### 3.1 Set up the NlpEngine and Hugging Face NER Recognizer
The NLP engine uses spaCy for text processing (lemmas, tokens, etc.). The Hugging Face model is added as a direct recognizer (HuggingFaceNerRecognizer) instead of being part of the NLP engine. This approach gives better control over model predictions.

In [15]:
# # Map OpenMed model entities to Presidio entities
# mapping = {
#     # Personal Info - Names & Demographics
#     "first_name": "PERSON",
#     "last_name": "PERSON",
#     "age": "AGE",
#     "gender": "GENDER",
#     "date_of_birth": "DATE_TIME",
#     "blood_type": "BLOOD_TYPE",
#     "occupation": "OCCUPATION",
#     "education_level": "EDUCATION_LEVEL",
#     "employment_status": "EMPLOYMENT_STATUS",
#     "language": "LOCATION",
#     "race_ethnicity": "ETHNICITY",
#     "sexuality": "SEXUALITY",
#     "political_view": "ORGANIZATION",
#     "religious_belief": "ORGANIZATION",
#     "biometric_identifier": "ID",
#     "pin": "ID",
#     # Identifiers (20+ types)
#     "account_number": "ID",
#     "customer_id": "ID",
#     "employee_id": "ID",
#     "unique_id": "ID",
#     "bank_routing_number": "US_BANK_NUMBER",
#     "swift_bic": "SWIFT_CODE",
#     "certificate_license_number": "MEDICAL_LICENSE",
#     "medical_record_number": "MEDICAL_LICENSE",
#     "health_plan_beneficiary_number": "ID",
#     "credit_debit_card": "CREDIT_CARD",
#     "cvv": "CVV",
#     "ssn": "US_SSN",
#     "tax_id": "ID",
#     "license_plate": "LICENSE_PLATE",
#     "vehicle_identifier": "ID",
#     "mac_address": "ID",
#     "device_identifier": "ID",
#     # Authentication & Security
#     "password": "PASSWORD",
#     "user_name": "USER_NAME",
#     # Contact Info (4 types)
#     "email": "EMAIL_ADDRESS",
#     "phone_number": "PHONE_NUMBER",
#     "fax_number": "PHONE_NUMBER",
#     "url": "URL",
#     # Location (6 types)
#     "city": "LOCATION",
#     "country": "LOCATION",
#     "county": "LOCATION",
#     "state": "LOCATION",
#     "street_address": "LOCATION",
#     "coordinate": "LOCATION",
#     "postcode": "ZIP_CODE",
#     # Network Info
#     "ipv4": "IP_ADDRESS",
#     "ipv6": "IP_ADDRESS",
#     # Temporal (3 types)
#     "date": "DATE_TIME",
#     "date_time": "DATE_TIME",
#     "time": "DATE_TIME",
#     # Organization (1 type)
#     "company_name": "ORGANIZATION",
# }

models = [
    {
        "lang_code": "en",
        "model_name": {
            "spacy": "en_core_web_sm",  # use a small spaCy model for lemmas, tokens etc.
            "transformers": "OpenMed/OpenMed-PII-BioClinicalModern-Large-395M-v1",
        },
    }
]

ner_config = NerModelConfiguration(
    #model_to_presidio_entity_mapping=mapping, 
    aggregation_strategy="simple"
)

# Create the NLP engine without entity mapping (will be handled by CanonicalMapper)
nlp_engine = TransformersNlpEngine(models=models, ner_model_configuration=ner_config)
nlp_engine.load()

Device set to use cpu


### 3.2 Set up the relevant recognizers
Add and remove recognizers to fit the dataset in hand. 
Removing all the recognizers that don't map to entities in our dataset.

In [16]:
def get_titles_recognizer():
    titles_recognizer = PatternRecognizer(
        deny_list=["Mr.", "Mrs.", "Ms.", "Miss", "Dr.", "Prof."],
        supported_entity="TITLE",
        name="TitlesRecognizer",
    )
    return titles_recognizer


def get_years_recognizer():
    years_recognizer = PatternRecognizer(
        patterns=[Pattern("YEAR", r"\b(19|20)\d{2}\b", score=0.1)],
        supported_entity="DATE_TIME",
        name="YearsRecognizer",
        context=["year", "at", "date", "in", "on"],
    )
    return years_recognizer


def get_age_recognizer():
    weak_regex = r"\b(110|[1-9]?[0-9])\b"
    age_pattern = Pattern(name="age (very weak)", regex=weak_regex, score=0.01)
    age_recognizer = PatternRecognizer(
        supported_entity="AGE",
        patterns=[age_pattern],
        name="AgeRecognizer",
        context=["month", "old", "turn", "age", "y/o"],
    )
    return age_recognizer


# Create Recognizer Registry
registry = RecognizerRegistry()
registry.load_predefined_recognizers(nlp_engine=nlp_engine)

# Add custom pattern recognizers
registry.add_recognizer(get_titles_recognizer())
registry.add_recognizer(get_years_recognizer())
registry.add_recognizer(get_age_recognizer())

# Remove unnecessary recognizers from presidio
unnecessary = [
    "NhsRecognizer",
    "UkNinoRecognizer",
    "SgFinRecognizer",
    "AuAbnRecognizer",
    "AuAcnRecognizer",
    "AuTfnRecognizer",
    "AuMedicareRecognizer",
    "InPanRecognizer",
    "InAadhaarRecognizer",
    "InVehicleRegistrationRecognizer",
    "InPassportRecognizer",
    "InVoterRecognizer",
    "UsLicenseRecognizer",
    "CryptoRecognizer",
    "SpacyRecognizer",
]

for rec in unnecessary:
    registry.remove_recognizer(rec)

### 3.3 Configure the context mechanism
Configure the `LemmaContextAawareEnhancer` which uses surrounding words to increase confidence in detection

In [17]:
# Set up the context aware enhancer
context_enhancer = LemmaContextAwareEnhancer(
    context_prefix_count=10, context_suffix_count=10
)

### 3.4 Create the AnalyzerEngine object

In [18]:
# Set up the engine, loads the NLP module (spaCy model by default)
# and other PII recognizers
analyzer_engine = AnalyzerEngine(
    nlp_engine=nlp_engine,
    context_aware_enhancer=context_enhancer,
    registry=registry,
    default_score_threshold=0.3,
)

# analyzer_engine = AnalyzerEngine()

pprint("Supported entities for English:")
pprint(analyzer_engine.get_supported_entities("en"), compact=True)

print("\nLoaded recognizers for English:")
pprint(
    [
        rec.name
        for rec in analyzer_engine.registry.get_recognizers("en", all_fields=True)
    ],
    compact=True,
)

print("\nLoaded Context Aware Enhancer:")
print(analyzer_engine.context_aware_enhancer.__class__.__name__)
pprint(json.dumps(analyzer_engine.context_aware_enhancer.__dict__), compact=True)


print("\nLoaded NER models:")
pprint(analyzer_engine.nlp_engine.models)

'Supported entities for English:'
['US_BANK_NUMBER', 'PERSON', 'US_SSN', 'MEDICAL_LICENSE', 'IBAN_CODE',
 'CREDIT_CARD', 'US_PASSPORT', 'ORGANIZATION', 'EMAIL', 'IP_ADDRESS', 'TITLE',
 'US_ITIN', 'NRP', 'AGE', 'ID', 'EMAIL_ADDRESS', 'PHONE_NUMBER', 'URL',
 'MAC_ADDRESS', 'DATE_TIME', 'LOCATION']

Loaded recognizers for English:
['CreditCardRecognizer', 'UsBankRecognizer', 'UsItinRecognizer',
 'UsPassportRecognizer', 'UsSsnRecognizer', 'DateRecognizer', 'EmailRecognizer',
 'IbanRecognizer', 'IpRecognizer', 'MedicalLicenseRecognizer',
 'MacAddressRecognizer', 'PhoneRecognizer', 'UrlRecognizer',
 'TransformersRecognizer', 'TitlesRecognizer', 'YearsRecognizer',
 'AgeRecognizer']

Loaded Context Aware Enhancer:
LemmaContextAwareEnhancer
('{"context_similarity_factor": 0.35, "min_score_with_context_similarity": '
 '0.4, "context_prefix_count": 10, "context_suffix_count": 10, '
 '"context_matching_mode": "substring"}')

Loaded NER models:
[{'lang_code': 'en',
  'model_name': {'spacy': 'en_cor

In [19]:
# Test Analyzer
text = "Yesterday in Mt. Sinai AP: Dana Silver, 79 years old female was complaining of stomach pain. Her ID is 154555"
res = analyzer_engine.analyze(text=text, language="en", return_decision_process=True)
for result in res:
    print(
        f"\nEntity: {result.entity_type}, Text: {text[result.start : result.end]}\n\nAnalysis explanation:"
    )
    pprint(result.analysis_explanation)


Entity: AGE, Text: 79

Analysis explanation:
{'recognizer': 'AgeRecognizer', 'pattern_name': 'age (very weak)', 'pattern': '\\b(110|[1-9]?[0-9])\\b', 'original_score': 0.01, 'score': 0.4, 'textual_explanation': 'Detected by `AgeRecognizer` using pattern `age (very weak)`', 'score_context_improvement': 0.39, 'supportive_context_word': 'old', 'validation_result': None, 'regex_flags': regex.I|M|S}


In [20]:
# Wrap the analyzer
wrapped_analyzer = PresidioAnalyzerWrapper(
    analyzer_engine=analyzer_engine,
    score_threshold=analyzer_engine.default_score_threshold,
    language="en",
)

## 4. Run predictions

Run the model on the full dataset. Returns a 5-column DataFrame (`sentence_id`, `token`, `annotation`, `prediction`, `start_indices`).

In [21]:
%%time
results_df = wrapped_analyzer.predict_dataset(dataset)
results_df.head()

CPU times: user 1min 49s, sys: 60 s, total: 2min 49s
Wall time: 2min


,sentence_id,token,annotation,prediction,start_indices
0,0,The,O,O,0
1,0,address,O,O,4
2,0,of,O,O,12
3,0,Persint,ORGANIZATION,O,15
4,0,is,O,O,23


## 5. Review entity mapping

`CanonicalMapper` auto-resolves entity labels from the results DataFrame using a two-phase workflow:

1. **Identify** - each label is matched to a canonical entity (exact alias, country-prefix, or fuzzy match)
2. **Project** - labels are projected onto the *canonical surface*, a set of entities at the majority-vote
   depth computed from the dataset annotations

After calling `analyze()`, inspect issues with `render_html()` or `get_issues()`. Issue types:

| Type | Severity | Meaning |
|------|----------|---------|
| `UNRESOLVED` | ERROR | Label could not be matched - must map or suppress |
| `COLLISION_AMBIGUOUS` | WARNING | Depth-2 label maps to multiple depth-3 entities - must pick one |
| `COLLISION_CROSS_BRANCH` | WARNING | Label maps across hierarchy branches |
| `PREDICTION_ONLY` | WARNING | Label only appears in predictions, not dataset - must map or suppress |
| `COLLISION_TRIVIAL` | INFO | Auto-fixed: descendant collapsed to single ancestor |
| `DATASET_ONLY` | INFO | Label only in dataset annotations (model never predicts it) |

`DATASET_ONLY` is **non-blocking** (INFO severity) - you can call `get_mapped_results_dataframe()`
even when `DATASET_ONLY` issues exist. They simply indicate entities the model never predicts.

Use `map({label: canonical})` to remap, or `map({label: None})` to suppress a label.
Pre-map labels you already know about *before* `analyze()` to avoid them appearing as issues.


In [22]:
# Analyze the results DataFrame.
# No constructor arguments needed — canonical depth is inferred automatically
# from the annotation labels in the DataFrame.
mapper = CanonicalMapper()
mapper.analyze(results_df)
mapper.render_html()


Label,Canonical representation,Status,Tokens
TITLE,PERSON (via TITLE)TITLE is a depth-4+ node; auto-projected up to its depth-3 canonical representative PERSON.,auto-mapped,142
STREET_ADDRESS,ADDRESSExact alias match in the entity hierarchy.,mapped,3071
PERSON,PERSONExact alias match in the entity hierarchy.,mapped,1369
GPE,GPEExact alias match in the entity hierarchy.,mapped,521
ORGANIZATION,ORGANIZATIONExact alias match in the entity hierarchy.,mapped,504
PHONE_NUMBER,PHONE_NUMBERExact alias match in the entity hierarchy.,mapped,350
DATE_TIME,DATE_TIMEExact alias match in the entity hierarchy.,mapped,219
CREDIT_CARD,FINANCIALExact alias match in the entity hierarchy.,mapped,136
US_SSN,SSNExact alias match in the entity hierarchy.,mapped,80
AGE,AGEExact alias match in the entity hierarchy.,mapped,74


In [23]:
# Review detected issues programmatically
severity_icons = {'error': 'ERROR', 'warning': 'WARNING', 'info': 'INFO'}
for issue in mapper.get_issues():
    icon = severity_icons[issue.severity.value]
    labels_str = ', '.join(issue.labels)
    print(f'[{icon}] {issue.type.value}: {labels_str}')
    if issue.annotation_count or issue.prediction_count:
        print(f'  annotations={issue.annotation_count}, predictions={issue.prediction_count}')
    print()


[WARNING] prediction_only: URL
  annotations=0, predictions=37

[INFO] collision_trivial: TITLE
  annotations=142, predictions=68

[INFO] dataset_only: STREET_ADDRESS, ZIP_CODE
  annotations=3121, predictions=0

[INFO] dataset_only: GPE
  annotations=521, predictions=0

[INFO] dataset_only: ORGANIZATION
  annotations=504, predictions=0

[INFO] dataset_only: NRP
  annotations=55, predictions=0

[INFO] dataset_only: DOMAIN_NAME
  annotations=37, predictions=0

[INFO] dataset_only: US_DRIVER_LICENSE
  annotations=9, predictions=0



In [24]:
# Resolve all WARNING+ issues programmatically (batch resolution pattern).
# map() re-evaluates issues automatically, so no need to call analyze() again.
#
# PREDICTION_ONLY (WARNING): labels the model predicts but the dataset never annotates.
#   -> suppress with None to exclude from evaluation
# COLLISION_AMBIGUOUS (WARNING): depth-2 labels that match multiple depth-3 entities.
#   -> suppress or remap to the most appropriate depth-3 entity
# UNRESOLVED (ERROR): no canonical match found.
#   -> must map to a canonical entity or suppress
# DATASET_ONLY (INFO): non-blocking, no action needed.
#
resolutions = {}
for issue in mapper.get_issues():
    if issue.severity.value in ("warning", "error"):
        for lbl in issue.labels:
            resolutions[lbl] = None  # suppress — adjust as needed for your model

if resolutions:
    mapper.map(resolutions)

# Get the remapped DataFrame — succeeds now that no blocking issues remain.
# Includes original_annotation and original_prediction columns for traceability.
mapped_df = mapper.get_mapped_results_dataframe()
mapper.render_html()


Label,Canonical representation,Status,Tokens
TITLE,PERSON (via TITLE)TITLE is a depth-4+ node; auto-projected up to its depth-3 canonical representative PERSON.,auto-mapped,142
STREET_ADDRESS,ADDRESSExact alias match in the entity hierarchy.,mapped,3071
PERSON,PERSONExact alias match in the entity hierarchy.,mapped,1369
GPE,GPEExact alias match in the entity hierarchy.,mapped,521
ORGANIZATION,ORGANIZATIONExact alias match in the entity hierarchy.,mapped,504
PHONE_NUMBER,PHONE_NUMBERExact alias match in the entity hierarchy.,mapped,350
DATE_TIME,DATE_TIMEExact alias match in the entity hierarchy.,mapped,219
CREDIT_CARD,FINANCIALExact alias match in the entity hierarchy.,mapped,136
US_SSN,SSNExact alias match in the entity hierarchy.,mapped,80
AGE,AGEExact alias match in the entity hierarchy.,mapped,74


## 6. Evaluate

In [25]:
# Set up the experiment tracker and log model + dataset params
experiment = get_experiment_tracker()
params = {"dataset_name": dataset_name, "model_name": wrapped_analyzer.name}
params.update(wrapped_analyzer.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)
experiment.log_parameter("entity_mappings", json.dumps(mapper.get_mapping()))

evaluator = SpanEvaluator(iou_threshold=0.75)

skip words not provided, using default skip words. If you want the evaluation to not use skip words, pass skip_words=[]


In [26]:
results = evaluator.calculate_score_on_df(results_df=mapped_df)

# Global PII precision/recall/F (all entity types collapsed to PII vs O)

# Track experiment results
experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, labels=entities)
experiment.end()

# Note that the experiment params and metrics are saved locally


saving experiment data to /Users/omrimendels/Documents/github/presidio-research/notebooks/experiment_20260428-195343.json


## 7. Evaluate results

In [27]:
# Plot output
plotter = Plotter(
    results=results, model_name=wrapped_analyzer.name, display_mode="interactive", beta=2
)
plotter.plot_scores()

In [28]:
pprint(
    {
        "PII F": results.pii_f,
        "PII recall": results.pii_recall,
        "PII precision": results.pii_precision,
    }
)

{'PII F': 0.2276138069034517,
 'PII precision': 0.6134831460674157,
 'PII recall': 0.19668587896253603}


## 8. Error analysis

The `ModelError` class groups similar errors together, finding all the false positive and false negative predictions where the model's behavior is consistent. 

This can help in detecting patterns in the model's performance, such as:
- Consistent labeling of specific words or patterns  
- Systematic overprediction or underprediction

**Note:** All errors are displayed using the **mapped model entity types** (e.g., `LOCATION`, `PERSON`) rather than the original dataset entity types (e.g., `STREET_ADDRESS`, `GPE`). This ensures consistency with how the model was evaluated.

In [29]:
plotter.plot_confusion_matrix(entities=entities, confmatrix=confmatrix)

In [30]:
plotter.plot_most_common_tokens()

### 8a. False positives
#### Most common false positive tokens:

In [31]:
ModelError.most_common_fp_tokens(results.model_errors)

Most common false positive tokens:
[('8 16', 18),
 ('dr.', 10),
 ('ms.', 9),
 ('501(c)3', 5),
 ('2270 66 1551', 2),
 ('ms', 2),
 ('060426070011', 2),
 ('2000 04 16', 1),
 ('063 966', 1),
 ('2006 04 07', 1)]
---------------
Example sentence with each FP token:
	- 8 16 (`8 16` pred as AGE)
	- Dr. (`dr.` pred as PERSON)
	- Ms. (`ms.` pred as PERSON)
	- 501(c)3 (`501(c)3` pred as AGE)
	- 2270 - 66 - 1551 (`2270 66 1551` pred as PHONE_NUMBER)
	- ms . (`ms` pred as PERSON)
	- 060426070011 (`060426070011` pred as PHONE_NUMBER)
	- 2000 - 04 - 16 (`2000 04 16` pred as DATE_TIME)
	- 063 966 (`063 966` pred as PHONE_NUMBER)
	- 2006 - 04 - 07 (`2006 04 07` pred as DATE_TIME)


[('8 16', 18),
 ('dr.', 10),
 ('ms.', 9),
 ('501(c)3', 5),
 ('2270 66 1551', 2),
 ('ms', 2),
 ('060426070011', 2),
 ('2000 04 16', 1),
 ('063 966', 1),
 ('2006 04 07', 1)]

#### More FP analysis

In [32]:
fps_df = ModelError.get_fps_dataframe(results.model_errors, entity="AGE")
fps_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,501(c)3,501(c)3,O,AGE
1,8 16,8 16,O,AGE
2,92 82 32,92 82 32,O,AGE
3,8 16,8 16,O,AGE
4,8 16,8 16,O,AGE
5,49,49,O,AGE
6,20,20,O,AGE
7,501(c)3,501(c)3,O,AGE
8,501(c)3,501(c)3,O,AGE
9,13,13,O,AGE


### 8b. False negatives (FN)

#### Most common false negative examples + a few samples with FN

In [33]:
ModelError.most_common_fn_tokens(results.model_errors, n=15)

Most common false negative tokens:
[('estonia', 9),
 ('norway', 8),
 ('greenlander', 7),
 ('canada', 6),
 ('american', 6),
 ('czech republic', 5),
 ('tunisia', 4),
 ('tuesday', 4),
 ('france', 4),
 ('russian', 4),
 ('united states', 4),
 ('iceland', 4),
 ('hungary', 4),
 ('czech', 4),
 ('thursday', 4)]
---------------
Example sentence with each FN token:
	- Estonia (`estonia` annotated as GPE)
	- Norway (`norway` annotated as GPE)
	- Greenlander (`greenlander` annotated as GPE)
	- Canada (`canada` annotated as GPE)
	- American (`american` annotated as NATIONALITY)
	- Czech Republic (`czech republic` annotated as GPE)
	- Tunisia (`tunisia` annotated as GPE)
	- Tuesday (`tuesday` annotated as DATE_TIME)
	- France (`france` annotated as GPE)
	- Russian (`russian` annotated as NATIONALITY)
	- United States (`united states` annotated as GPE)
	- Iceland (`iceland` annotated as GPE)
	- Hungary (`hungary` annotated as GPE)
	- Czech (`czech` annotated as NATIONALITY)
	- Thursday (`thursday` ann

[('estonia', 9),
 ('norway', 8),
 ('greenlander', 7),
 ('canada', 6),
 ('american', 6),
 ('czech republic', 5),
 ('tunisia', 4),
 ('tuesday', 4),
 ('france', 4),
 ('russian', 4),
 ('united states', 4),
 ('iceland', 4),
 ('hungary', 4),
 ('czech', 4),
 ('thursday', 4)]

#### More FN analysis

In [34]:
fns_df = ModelError.get_fns_dataframe(results.model_errors, entity="PERSON")

In [35]:
fns_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,Krisztián Szöllösy,krisztián szöllösy,PERSON,O
1,Szabina J Gelencsér,szabina j gelencsér,PERSON,O
2,Rubija,rubija,PERSON,O
3,Faina D. Yefremova Yefremova,faina d. yefremova yefremova,PERSON,O
4,Johannes Varvio,johannes varvio,PERSON,O
5,Šárka Ottová,šárka ottová,PERSON,O
6,Christiansen,christiansen,PERSON,O
7,Sara Schwarz,sara schwarz,PERSON,O
8,Ubul Nicole Mary John,ubul nicole mary john,PERSON,O
9,Dr. Cettina Fanucci,dr. cettina fanucci,PERSON,O
